# Bauabrechnung Glasfaser-Tiefbau – Aufmaß, Abrechnung und GAEB X31
**Construction billing for a fibre-optic civil works project – quantity survey (Aufmaß), billing against the bill of quantities and GAEB X31 export**

Fictitious example project (640 m trench in a street). All quantities and prices are assumptions.
Steps: 1 bill of quantities (LV) · 2 quantity survey (Aufmaß, REB-VB 23.003 principle) · 3 billing and over/under quantities · 4 GAEB X31 export, Excel, charts and report.

In [ ]:
import os, json, datetime
import pandas as pd
import matplotlib.pyplot as plt
from xml.etree import ElementTree as ET
os.makedirs('results', exist_ok=True); os.makedirs('images', exist_ok=True)
pd.set_option('display.width', 160)

## 1 Bill of quantities (Leistungsverzeichnis, LV)

In [ ]:
lv = pd.DataFrame([
 ('01','01.0010','Baustelle einrichten, vorhalten und räumen',1,'psch',2800.00),
 ('01','01.0020','Verkehrssicherung nach RSA 21 vorhalten',4,'Wo',450.00),
 ('02','02.0010','Asphaltbefestigung schneiden, d bis 12 cm',660,'m',4.20),
 ('02','02.0020','Asphalt aufnehmen und entsorgen, d bis 12 cm',192,'m²',14.50),
 ('02','02.0030','Gehwegplatten aufnehmen und seitlich lagern',150,'m²',9.80),
 ('02','02.0040','Leitungsgraben ausheben, Homogenbereich B1, Tiefe bis 0,80 m',230,'m³',38.00),
 ('02','02.0050','Kabelschutzrohr DN 50 (Speedpipe) liefern und verlegen',640,'m',6.90),
 ('02','02.0060','Leitungszone mit Sand 0/2 herstellen',60,'m³',42.00),
 ('02','02.0070','Graben verfüllen und lagenweise verdichten',170,'m³',22.00),
 ('02','02.0080','Überschüssigen Boden laden und entsorgen',60,'m³',31.00),
 ('02','02.0090','Kopfloch für Hausanschluss herstellen und verfüllen',24,'St',145.00),
 ('03','03.0010','Frostschutzschicht 0/32 einbauen, d = 30 cm',192,'m²',11.50),
 ('03','03.0020','Asphalttragdeckschicht AC 16 TD einbauen, d = 10 cm',192,'m²',34.00),
 ('03','03.0030','Gehwegplatten wieder verlegen',150,'m²',24.00),
], columns=['Titel','OZ','Kurztext','Menge_LV','Einheit','EP'])
titel = {'01':'Baustelleneinrichtung und Verkehrssicherung','02':'Erdarbeiten und Leitungsbau','03':'Oberflächenwiederherstellung'}
lv['GP_LV'] = (lv.Menge_LV * lv.EP).round(2)
print('LV netto:', lv.GP_LV.sum())
lv

## 2 Quantity survey (Aufmaß) – REB-VB 23.003 principle
Every line: number × length × width × height; deductions with negative number. Sections: A 0+000–0+150 road · B 0+150–0+310 footway · C 0+310–0+480 road · D 0+480–0+640 footway.

In [ ]:
A = [  # OZ, Ort/Station, Rechenansatz, Anzahl, L, B, H
 ('01.0010','gesamte Baustelle','pauschal',1,None,None,None),
 ('01.0020','0+000–0+640','Bauzeit 5 Wochen',5,None,None,None),
 ('02.0010','A 0+000–0+150','2 Schnitte längs',2,150,None,None),
 ('02.0010','C 0+310–0+480','2 Schnitte längs',2,170,None,None),
 ('02.0010','A, C','Querschnitte Anfang/Ende',4,0.65,None,None),
 ('02.0020','A 0+000–0+150','L × Aufbruchbreite',1,150,0.65,None),
 ('02.0020','C 0+310–0+480','L × Aufbruchbreite',1,170,0.65,None),
 ('02.0030','B 0+150–0+310','L × Breite',1,160,0.50,None),
 ('02.0030','D 0+480–0+640','L × Breite',1,160,0.50,None),
 ('02.0040','A 0+000–0+150','L × Grabenbreite × Tiefe',1,150,0.40,0.85),
 ('02.0040','B 0+150–0+310','L × Grabenbreite × Tiefe',1,160,0.40,0.80),
 ('02.0040','C 0+310–0+480','L × Grabenbreite × Tiefe',1,170,0.40,0.80),
 ('02.0040','D 0+480–0+640','L × Grabenbreite × Tiefe',1,160,0.40,0.75),
 ('02.0050','0+000–0+640','Trassenlänge',1,640,None,None),
 ('02.0050','Kopflöcher','Zuschlag Bögen/Anschluss',26,0.50,None,None),
 ('02.0060','0+000–0+640','L × Grabenbreite × Höhe Leitungszone',1,640,0.40,0.25),
 ('02.0070','A 0+000–0+150','Aushub',1,150,0.40,0.85),
 ('02.0070','B 0+150–0+310','Aushub',1,160,0.40,0.80),
 ('02.0070','C 0+310–0+480','Aushub',1,170,0.40,0.80),
 ('02.0070','D 0+480–0+640','Aushub',1,160,0.40,0.75),
 ('02.0070','A–D','abzgl. Leitungszone',-1,640,0.40,0.25),
 ('02.0080','A–D','verdrängtes Volumen Leitungszone',1,640,0.40,0.25),
 ('02.0090','lt. Bestandsplan','Hausanschlüsse Nr. 1–26',26,None,None,None),
 ('03.0010','A, C','wie Pos. 02.0020',1,320,0.65,None),
 ('03.0020','A, C','wie Pos. 02.0020',1,320,0.65,None),
 ('03.0030','B, D','wie Pos. 02.0030',1,320,0.50,None),
]
aufmass = pd.DataFrame(A, columns=['OZ','Ort','Rechenansatz','Anzahl','Laenge','Breite','Hoehe'])
aufmass['Menge'] = (aufmass.Anzahl * aufmass[['Laenge','Breite','Hoehe']].fillna(1).prod(axis=1)).round(3)
aufmass.insert(1, 'Zeile', aufmass.groupby('OZ').cumcount() + 1)
aufmass

## 3 Billing and over/under quantities (Mehr- und Mindermengen)

In [ ]:
menge = aufmass.groupby('OZ', as_index=False).Menge.sum().rename(columns={'Menge':'Menge_Aufmass'})
ab = lv.merge(menge, on='OZ')
ab['GP_Abrechnung'] = (ab.Menge_Aufmass * ab.EP).round(2)
ab['Diff_Menge'] = (ab.Menge_Aufmass - ab.Menge_LV).round(3)
ab['Diff_EUR'] = (ab.GP_Abrechnung - ab.GP_LV).round(2)
ab['Diff_%'] = (ab.Diff_Menge / ab.Menge_LV * 100).round(1)
ab['Hinweis'] = ab['Diff_%'].abs().gt(10).map({True:'> ±10 % – VOB/B § 2 Abs. 3 prüfen', False:''})
net_lv, net_ab = ab.GP_LV.sum().round(2), ab.GP_Abrechnung.sum().round(2)
k = {
 'Auftragssumme LV netto (EUR)': net_lv,
 'Abrechnungssumme netto (EUR)': net_ab,
 'Differenz netto (EUR)': round(net_ab - net_lv, 2),
 'Differenz (%)': round((net_ab - net_lv) / net_lv * 100, 1),
 'MwSt 19 % (EUR)': round(net_ab * 0.19, 2),
 'Abrechnungssumme brutto (EUR)': round(net_ab * 1.19, 2),
 'Positionen mit Mehrmenge': int((ab.Diff_Menge > 0).sum()),
 'Positionen mit Mindermenge': int((ab.Diff_Menge < 0).sum()),
 'Positionen > ±10 % (VOB/B § 2 Abs. 3)': int(ab['Diff_%'].abs().gt(10).sum()),
}
kz = pd.Series(k, name='Wert', dtype=object); kz.index.name = 'Kennzahl'
print(kz.to_string())
ab[['OZ','Kurztext','Einheit','Menge_LV','Menge_Aufmass','Diff_%','EP','GP_LV','GP_Abrechnung','Diff_EUR','Hinweis']]

## 4 Results: Excel, CSV, GAEB X31, charts

In [ ]:
tsum = ab.groupby('Titel')[['GP_LV','GP_Abrechnung']].sum().round(2)
tsum.insert(0, 'Bezeichnung', [titel[t] for t in tsum.index])
with pd.ExcelWriter('results/bauabrechnung_glasfaser_tiefbau.xlsx') as xw:
    kz.to_frame().to_excel(xw, sheet_name='Kennzahlen')
    lv.to_excel(xw, sheet_name='LV', index=False)
    aufmass.to_excel(xw, sheet_name='Aufmass', index=False)
    ab.to_excel(xw, sheet_name='Abrechnung', index=False)
    tsum.to_excel(xw, sheet_name='Titelsummen')
lv.to_csv('results/lv.csv', index=False, sep=';', decimal=',')
aufmass.to_csv('results/aufmass.csv', index=False, sep=';', decimal=',')
ab.to_csv('results/abrechnung.csv', index=False, sep=';', decimal=',')
kz.to_csv('results/kennzahlen.csv', sep=';', decimal=',')
print('saved')

In [ ]:
# GAEB DA XML 3.3 – DA 31 (quantity determination) – simplified structure, not validated against the official XSD
NS = 'http://www.gaeb.de/GAEB_DA_XML/DA31/3.3'
ET.register_namespace('', NS)
q = lambda t: f'{{{NS}}}{t}'
g = ET.Element(q('GAEB'))
info = ET.SubElement(g, q('GAEBInfo'))
for t, v in (('Version','3.3'),('Date',datetime.date(2026,11,24).isoformat()),('ProgSystem','Python / pandas (portfolio example)')):
    ET.SubElement(info, q(t)).text = v
ET.SubElement(ET.SubElement(g, q('PrjInfo')), q('NamePrj')).text = 'Glasfaser-Tiefbau Musterstrasse (Beispiel)'
award = ET.SubElement(g, q('Award')); ET.SubElement(award, q('DP')).text = '31'
boq = ET.SubElement(ET.SubElement(award, q('BoQ')), q('BoQBody'))
for t, grp in ab.groupby('Titel'):
    ctg = ET.SubElement(boq, q('BoQCtgy'), RNoPart=t)
    ET.SubElement(ET.SubElement(ctg, q('LblTx')), q('p')).text = titel[t]
    items = ET.SubElement(ET.SubElement(ctg, q('BoQBody')), q('Itemlist'))
    for _, r in grp.iterrows():
        it = ET.SubElement(items, q('Item'), RNoPart=r.OZ.split('.')[1])
        ET.SubElement(it, q('Qty')).text = f'{r.Menge_Aufmass:.3f}'
        ET.SubElement(it, q('QU')).text = r.Einheit
        ET.SubElement(ET.SubElement(ET.SubElement(it, q('Description')), q('CompleteText')), q('OutlineText')).text = r.Kurztext
        qd = ET.SubElement(it, q('QtyDeterm'))
        for _, z in aufmass[aufmass.OZ == r.OZ].iterrows():
            ln = ET.SubElement(qd, q('QDetermItem'))
            ET.SubElement(ln, q('Text')).text = f'{z.Ort}: {z.Rechenansatz}'
            fac = [f'{z.Anzahl:g}'] + [f'{v:g}' for v in (z.Laenge, z.Breite, z.Hoehe) if pd.notna(v)]
            ET.SubElement(ln, q('Formula')).text = ' * '.join(fac)
            ET.SubElement(ln, q('Result')).text = f'{z.Menge:.3f}'
ET.indent(g)
ET.ElementTree(g).write('results/aufmass_glasfaser_tiefbau.X31', encoding='utf-8', xml_declaration=True)
print(open('results/aufmass_glasfaser_tiefbau.X31', encoding='utf-8').read()[:600])

In [ ]:
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
C_LV, C_AB, C_UP, C_DN = '#9DB4D3', '#1F3864', '#C0392B', '#2E8B57'
lab = ab.OZ + '  ' + ab.Kurztext.str.slice(0, 34)

fig, ax = plt.subplots(figsize=(11, 6.5))
y = range(len(ab))
ax.barh([i + 0.2 for i in y], ab.GP_LV, 0.4, color=C_LV, label='LV (Auftrag)')
ax.barh([i - 0.2 for i in y], ab.GP_Abrechnung, 0.4, color=C_AB, label='Abrechnung (Aufmaß)')
ax.set_yticks(list(y)); ax.set_yticklabels(lab, fontsize=8.5); ax.invert_yaxis()
ax.set_xlabel('EUR netto'); ax.legend(frameon=False, loc='lower right')
ax.set_title(f'LV {net_lv:,.0f} € → Abrechnung {net_ab:,.0f} € netto'.replace(',', '.'), loc='left', fontweight='bold')
plt.tight_layout(); plt.savefig('images/01_LV_vs_Abrechnung.png', dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(11, 6))
cols = [C_UP if v > 0 else C_DN if v < 0 else '#999999' for v in ab['Diff_%']]
ax.barh(lab, ab['Diff_%'], color=cols); ax.invert_yaxis()
for x in (-10, 10): ax.axvline(x, ls='--', lw=1, color='#555555')
ax.axvline(0, color='black', lw=0.8)
for i, v in enumerate(ab['Diff_%']):
    ax.text(v + (0.6 if v >= 0 else -0.6), i, f'{v:+.1f} %'.replace('.', ','), va='center', ha='left' if v >= 0 else 'right', fontsize=8.5)
ax.set_xlim(-26, 32); ax.set_xlabel('Mengenabweichung Aufmaß gegenüber LV (%)'); ax.tick_params(axis='y', labelsize=8.5)
ax.set_title('Mehr- und Mindermengen – gestrichelt: ±10 % (VOB/B § 2 Abs. 3)', loc='left', fontweight='bold')
plt.tight_layout(); plt.savefig('images/02_Mehr_Mindermengen.png', dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(8, 4.5))
x = range(len(tsum))
ax.bar([i - 0.2 for i in x], tsum.GP_LV, 0.4, color=C_LV, label='LV')
ax.bar([i + 0.2 for i in x], tsum.GP_Abrechnung, 0.4, color=C_AB, label='Abrechnung')
ax.set_xticks(list(x)); ax.set_xticklabels([f'{t} {titel[t]}'.replace(' und ', ' und\n', 1) for t in tsum.index], fontsize=8.5)
ax.set_ylabel('EUR netto'); ax.legend(frameon=False); ax.set_title('Summen je Titel', loc='left', fontweight='bold')
plt.tight_layout(); plt.savefig('images/03_Titelsummen.png', dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(11, 2.8))
secs = [('A', 0, 150, 'Fahrbahn', 0.85), ('B', 150, 310, 'Gehweg', 0.80), ('C', 310, 480, 'Fahrbahn', 0.80), ('D', 480, 640, 'Gehweg', 0.75)]
for s, a, b, t, d in secs:
    ax.add_patch(plt.Rectangle((a, 0), b - a, 1, color='#555555' if t == 'Fahrbahn' else '#C9B79C', alpha=0.85))
    ax.text((a + b) / 2, 0.5, f'{s}  {t}\n{b - a} m · Tiefe {d:.2f} m'.replace('.', ','), ha='center', va='center', color='white', fontsize=9, fontweight='bold')
for st in (0, 150, 310, 480, 640): ax.text(st, -0.18, f'0+{st:03d}', ha='center', fontsize=8.5)
ax.plot([0, 640], [1.12, 1.12], color='#E67E22', lw=3); ax.text(320, 1.22, 'Kabelschutzrohr DN 50 – 640 m + 26 Kopflöcher', ha='center', fontsize=9)
ax.set_xlim(-15, 655); ax.set_ylim(-0.35, 1.4); ax.axis('off')
ax.set_title('Aufmaß-Abschnitte (Stationierung)', loc='left', fontweight='bold')
plt.tight_layout(); plt.savefig('images/04_Aufmass_Abschnitte.png', dpi=150); plt.close()
print('charts saved')